# CLV Modeling: BG/NBD + Gamma-Gamma

This notebook fits the probabilistic CLV model on customer transaction history:

1. **BG/NBD model** — predicts future purchase frequency per customer
2. **Gamma-Gamma model** — predicts average transaction value per customer
3. **CLV = predicted purchases × expected spend** over 12-month horizon

**Key assumption:** One-time buyers (~88%) are NOT excluded — BG/NBD natively models them as having dropped out after their first purchase, which is the correct answer for that cohort.

**Outputs:**
- `models/bgnbd_model.pkl` — fitted BG/NBD model
- `models/gg_model.pkl` — fitted Gamma-Gamma model
- `data/processed/clv_scored.csv` — per-customer CLV predictions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.plotting import (
    plot_frequency_recency_matrix,
    plot_probability_alive_matrix,
    plot_period_transactions,
)
from scipy.stats import pearsonr

# Paths
DATA_PATH   = "../data/raw/clv_data.csv"
SCORED_PATH = "../data/processed/clv_scored.csv"
BGNBD_PATH  = "../models/bgnbd_model.pkl"
GG_PATH     = "../models/gg_model.pkl"

# CLV horizon
CLV_MONTHS = 12
CLV_DAYS   = 365

print("Libraries loaded.")

## 1. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} customers × {df.shape[1]} features")
print(f"\nBG/NBD core inputs:")
print(df[['frequency', 'recency', 'T', 'monetary_value']].describe().round(2))

In [ ]:
# One-time buyer breakdown
one_time = (df['frequency'] == 0).sum()
repeat   = (df['frequency'] >= 1).sum()
print(f"One-time buyers (frequency=0): {one_time:,} ({one_time/len(df):.1%})")
print(f"Repeat buyers   (frequency≥1): {repeat:,} ({repeat/len(df):.1%})")

# Frequency distribution
fig, ax = plt.subplots(figsize=(8, 4))
freq_counts = df['frequency'].value_counts().sort_index().head(15)
ax.bar(freq_counts.index, freq_counts.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of Repeat Purchases (frequency)')
ax.set_ylabel('Customer Count')
ax.set_title('Purchase Frequency Distribution')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 2. Fit BG/NBD Model

The **Beta-Geometric / Negative Binomial Distribution (BG/NBD)** model:
- Models each customer's purchase rate as Poisson with gamma-distributed heterogeneity
- Models dropout probability as beta-geometric process
- Outputs: probability a customer is still alive (`p_alive`) + predicted future transactions

**Parameters:** r, α (transaction rate distribution), a, b (dropout rate distribution)

In [ ]:
# Fit BG/NBD model
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(
    df['frequency'],
    df['recency'],
    df['T'],
    verbose=True
)

print("\n=== BG/NBD Model Summary ===")
print(bgf.summary)

In [ ]:
# Sanity check: all params should be positive
params = bgf.params_
for name, val in params.items():
    assert val > 0, f"BG/NBD param '{name}' = {val:.4f} — must be positive"
print("✓ All BG/NBD parameters are positive")
print(f"  r={params['r']:.4f}, α={params['alpha']:.4f}, a={params['a']:.4f}, b={params['b']:.4f}")

In [ ]:
# Frequency-Recency matrix: expected purchases in next 12 months
fig = plt.figure(figsize=(12, 5))
plot_frequency_recency_matrix(bgf, T=CLV_DAYS, max_frequency=15)
plt.title('Expected Purchases in Next 12 Months\n(by customer frequency and recency)')
plt.tight_layout()
plt.show()

In [ ]:
# P(alive) matrix
fig = plt.figure(figsize=(12, 5))
plot_probability_alive_matrix(bgf, max_frequency=15)
plt.title('Probability Customer is Still Active\n(by frequency and recency)')
plt.tight_layout()
plt.show()

In [ ]:
# Compute per-customer predictions
df['p_alive'] = bgf.conditional_probability_alive(
    df['frequency'], df['recency'], df['T']
)
df['predicted_purchases_12m'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    CLV_DAYS, df['frequency'], df['recency'], df['T']
)

print("=== BG/NBD Predictions ===")
print(df[['p_alive', 'predicted_purchases_12m']].describe().round(4))

## 3. Fit Gamma-Gamma Spend Model

The **Gamma-Gamma model** estimates average transaction value, conditional on being a repeat buyer.

**Key assumption:** Purchase frequency and monetary value are **independent** (correlation < 0.3 required).

In [ ]:
# Gamma-Gamma requires repeat buyers only
repeat_buyers = df[df['frequency'] > 0].copy()
print(f"Repeat buyers for Gamma-Gamma fit: {len(repeat_buyers):,}")

# Verify independence assumption: |corr(frequency, monetary_value)| < 0.3
corr, pval = pearsonr(repeat_buyers['frequency'], repeat_buyers['monetary_value'])
print(f"\nPearson r(frequency, monetary_value) = {corr:.4f}  (p={pval:.4f})")
assert abs(corr) < 0.3, f"Correlation {corr:.3f} violates Gamma-Gamma independence assumption (must be < 0.3)"
print("✓ Frequency/monetary independence assumption satisfied")

In [ ]:
# Fit Gamma-Gamma model
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(
    repeat_buyers['frequency'],
    repeat_buyers['monetary_value'],
)

print("=== Gamma-Gamma Model Summary ===")
print(ggf.summary)

In [ ]:
# Compute expected average spend per customer
# For repeat buyers: use Gamma-Gamma posterior estimate
# For one-time buyers (frequency=0): fallback to observed avg_order_value

gg_spend = ggf.conditional_expected_average_profit(
    repeat_buyers['frequency'],
    repeat_buyers['monetary_value'],
)

df['expected_avg_spend'] = df['avg_order_value']   # default: observed AOV
df.loc[df['frequency'] > 0, 'expected_avg_spend'] = gg_spend.values

print("=== Expected Average Spend ===")
print(df['expected_avg_spend'].describe().round(2))

## 4. Compute CLV

**CLV (12-month) = predicted_purchases_12m × expected_avg_spend**

This is undiscounted CLV for simplicity of stakeholder communication.

In [ ]:
# Compute CLV
df['clv_12m'] = df['predicted_purchases_12m'] * df['expected_avg_spend']

print("=== 12-Month CLV Distribution ===")
print(df['clv_12m'].describe().round(2))

# Sanity check: one-time buyers should have lower CLV than repeat buyers
clv_onetime = df.loc[df['frequency'] == 0, 'clv_12m'].mean()
clv_repeat  = df.loc[df['frequency'] >= 1, 'clv_12m'].mean()
print(f"\nAvg CLV — one-time buyers: ${clv_onetime:.2f}")
print(f"Avg CLV — repeat buyers:   ${clv_repeat:.2f}")
assert clv_onetime < clv_repeat, "One-time buyers should have lower CLV than repeat buyers"
print("✓ Sanity check passed: one-time buyer CLV < repeat buyer CLV")

In [ ]:
# CLV distribution (log scale)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Linear scale (clipped for clarity)
clv_clip = df['clv_12m'].clip(upper=df['clv_12m'].quantile(0.99))
axes[0].hist(clv_clip, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Predicted 12-Month CLV ($)')
axes[0].set_ylabel('Customer Count')
axes[0].set_title('CLV Distribution (clipped at 99th pct)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Log scale (all customers)
clv_nonzero = df.loc[df['clv_12m'] > 0, 'clv_12m']
axes[1].hist(np.log1p(clv_nonzero), bins=50, color='darkorange', edgecolor='white')
axes[1].set_xlabel('log(1 + CLV)')
axes[1].set_title('CLV Distribution (log scale, non-zero)')

plt.suptitle('12-Month Predicted Customer Lifetime Value', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue concentration (Pareto check)
df_sorted = df.sort_values('clv_12m', ascending=False).reset_index(drop=True)
df_sorted['cumulative_clv_pct'] = df_sorted['clv_12m'].cumsum() / df_sorted['clv_12m'].sum()
df_sorted['customer_pct'] = (df_sorted.index + 1) / len(df_sorted)

top20_revenue = df_sorted.loc[df_sorted['customer_pct'] <= 0.20, 'clv_12m'].sum()
total_revenue = df_sorted['clv_12m'].sum()
print(f"Top 20% customers → {top20_revenue/total_revenue:.1%} of predicted CLV")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_sorted['customer_pct'] * 100, df_sorted['cumulative_clv_pct'] * 100,
        color='steelblue', linewidth=2)
ax.axvline(20, color='red', linestyle='--', alpha=0.7, label='Top 20% customers')
ax.axhline(df_sorted.loc[df_sorted['customer_pct'] <= 0.20, 'cumulative_clv_pct'].max() * 100,
           color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('Cumulative % of Customers (ranked by CLV)')
ax.set_ylabel('Cumulative % of Predicted Revenue')
ax.set_title('Revenue Concentration (Predicted CLV)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Save Models and Scored Data

In [ ]:
# Save BG/NBD and Gamma-Gamma models
os.makedirs('../models', exist_ok=True)
joblib.dump(bgf, BGNBD_PATH)
joblib.dump(ggf, GG_PATH)
print(f"✓ BG/NBD model saved to {BGNBD_PATH}")
print(f"✓ Gamma-Gamma model saved to {GG_PATH}")

In [ ]:
# Save scored customer data
os.makedirs('../data/processed', exist_ok=True)
df.to_csv(SCORED_PATH, index=False)
print(f"✓ CLV scored data saved to {SCORED_PATH}")
print(f"  Shape: {df.shape[0]:,} customers × {df.shape[1]} columns")
print(f"\nNew columns added:")
new_cols = ['p_alive', 'predicted_purchases_12m', 'expected_avg_spend', 'clv_12m']
print(df[new_cols].describe().round(4))

---

**Next:** [03_clv_validation.ipynb](./03_clv_validation.ipynb) — Temporal holdout backtesting and lift curve validation